# Lab 05 · Simple Linear Regression
**CS-3105 Machine Learning Lab** · Lab Instructor: Engr. Muhammad Awais

**CLO 7** (C-3 Apply, PLO 4 Investigation): *Apply machine learning algorithms and assess their performance using appropriate evaluation metrics.*

**Goal:** build a complete price predictor for one car on real data: prepare the data, split it, train a simple linear regression with the least squares method, test it on unseen ads, and use it.

**Data:** `pakistan_used_cars_clean.csv`, the cleaned OLX Pakistan used-car ads from Lab 03 (Open Data Pakistan, 2021). Use the copy given with this lab so everyone has the same file.

**Rules:** label every output, e.g. `print("T5a:", ...)` · use `random_state=42` for the split · written answers go in Markdown cells · Runtime → Restart and run all before you submit · push to `ML-Lab-<RollNo>/Lab05/Lab05.ipynb` by the deadline announced in class (11:59 PM PKT).

Name: ZULQARNAIN MOHI UD DIN  Roll No: 2024-SE-07

### Setup (run first, do not edit)
Put `pakistan_used_cars_clean.csv` next to this notebook (Colab: folder icon → Upload).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
pd.set_option('display.width', 140)

## Task 1 · Load and explore the real data
- **(a)** Load `pakistan_used_cars_clean.csv` and print its shape.
- **(b)** Keep only **Corolla GLI**, **Cash** ads, `Year` 2005–2018 and `Price_lakh` between 5 and 40. Save it as `gli` and print the shape.
- **(c)** Summary statistics (`describe()`) of `Age` and `Price_lakh`.
- **(d)** The correlation between `Age` and `Price_lakh`.
- **(e)** A scatter plot of `Price_lakh` against `Age`, with a title and axis labels.

**Write (Markdown):** is a straight line a sensible model for this data? Use (d) and (e).

In [ ]:
# T1: load, filter, explore
import os, glob
CSV = "pakistan_used_cars_clean.csv"
if not os.path.exists(CSV):
    found = glob.glob("/content/*used_cars*.csv") + glob.glob("**/*used_cars*.csv", recursive=True)
    if found:
        CSV = found[0]
    else:
        try:   # Google Colab: ask for the file
            from google.colab import files
            up = files.upload()
            CSV = list(up.keys())[0]
        except ImportError:
            raise FileNotFoundError("Put pakistan_used_cars_clean.csv in the same folder as this notebook, then run again.")
df = pd.read_csv(CSV)
df.columns = [c.strip() for c in df.columns]
print("T1a: full data shape =", df.shape)

def col(name):   # find a column ignoring case/spaces/underscores
    key = name.lower().replace(" ", "").replace("_", "")
    for c in df.columns:
        if c.lower().replace(" ", "").replace("_", "") == key:
            return c
    raise KeyError("Column '%s' not found. Columns are: %s" % (name, list(df.columns)))

model_c, year_c, price_c = col("Model"), col("Year"), col("Price_lakh")
df[year_c] = pd.to_numeric(df[year_c], errors="coerce")
df[price_c] = pd.to_numeric(df[price_c], errors="coerce")
if "Age" not in df.columns:
    df["Age"] = 2020 - df[year_c]
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")

# payment column = the one that holds the value 'Cash'
pay_col = None
for c in df.columns:
    if df[c].dtype == object and df[c].astype(str).str.strip().str.lower().eq("cash").any():
        pay_col = c; break

mask = (df[model_c].astype(str).str.strip().str.lower() == "corolla gli")        & df[year_c].between(2005, 2018) & df[price_c].between(5, 40)
if pay_col is not None:
    mask &= df[pay_col].astype(str).str.strip().str.lower() == "cash"
else:
    print("WARNING: no 'Cash' column found, Cash filter skipped")
gli = df[mask].dropna(subset=["Age", price_c]).copy()
if price_c != "Price_lakh":
    gli["Price_lakh"] = gli[price_c]
print("T1b: gli shape =", gli.shape, "(payment column used:", pay_col, ")")

print("T1c: summary statistics")
print(gli[["Age", "Price_lakh"]].describe().round(2))

corr = gli["Age"].corr(gli["Price_lakh"])
print("T1d: correlation(Age, Price_lakh) = %.4f" % corr)

plt.figure(figsize=(7, 4.5))
plt.scatter(gli["Age"], gli["Price_lakh"], alpha=0.5, s=18, color="tab:blue")
plt.title("Corolla GLI (cash ads): price vs age")
plt.xlabel("Age (years)"); plt.ylabel("Price (lakh PKR)")
plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig("fig_t1_scatter.png", dpi=150); plt.show()

**Is a straight line sensible?** Yes, as a first model. The correlation in (d) is strongly negative, so older cars are clearly cheaper, and the scatter plot in (e) falls steadily from left to right in a roughly linear band. The points scatter around that band because mileage, condition, city and seller also move the price. The scatter is not a perfect line, and old cars may flatten out slightly, so a line is a good approximation but not the whole truth.

## Task 2 · Features, target and the train/test split
- **(a)** `X` = the `Age` column as a **2-D** table, `y` = `Price_lakh`. Print both shapes.
- **(b)** Split 80 % train / 20 % test with `train_test_split(..., test_size=0.2, random_state=42)`. Print the four shapes.

In [ ]:
# T2: features, target, split
X = gli[["Age"]]            # 2-D: DataFrame with one column
y = gli["Price_lakh"]       # 1-D Series
print("T2a: X shape =", X.shape, "| y shape =", y.shape)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("T2b: X_train", X_train.shape, "| X_test", X_test.shape,
      "| y_train", y_train.shape, "| y_test", y_test.shape)

## Task 3 · Least squares with NumPy (training data only)
Use only `X_train` and `y_train`, and NumPy (no scikit-learn):
- **(a)** The means x̄ and ȳ.
- **(b)** The slope b₁ = Σ(x − x̄)(y − ȳ) / Σ(x − x̄)² and the intercept b₀ = ȳ − b₁x̄.
- **(c)** **Write (Markdown):** what b₁ and b₀ mean for a Corolla GLI, with units.

In [ ]:
# T3: least squares by hand (training data only)
x = X_train["Age"].to_numpy()
t = y_train.to_numpy()
x_bar, y_bar = x.mean(), t.mean()
print("T3a: x_bar = %.4f years | y_bar = %.4f lakh" % (x_bar, y_bar))

b1 = np.sum((x - x_bar) * (t - y_bar)) / np.sum((x - x_bar) ** 2)
b0 = y_bar - b1 * x_bar
print("T3b: slope b1 = %.4f lakh/year | intercept b0 = %.4f lakh" % (b1, b0))
print("T3c: price = %.3f %+.3f * age  (lakh)" % (b0, b1))

**Meaning of b₁ and b₀.** **b₁** (slope, negative) is the average change in the price of a Corolla GLI for each extra year of age, in **lakh PKR per year**. A car one year older is expected to sell for about |b₁| lakh less. **b₀** (intercept) is the predicted price, in **lakh PKR**, of a car of age 0, a brand-new GLI. It is where the line meets the price axis and is an extrapolation beyond the youngest cars in the data (age 2 and up), so it should not be read as a real showroom price.

## Task 4 · Train the model with scikit-learn
- **(a)** Create a `LinearRegression()` model, fit it on the **training** data and print the slope (`coef_`) and intercept (`intercept_`).
- **(b)** Show with `np.isclose` that they equal your Task 3 values.

In [ ]:
# T4: scikit-learn model
model = LinearRegression()
model.fit(X_train, y_train)
print("T4a: coef_ = %.4f | intercept_ = %.4f" % (model.coef_[0], model.intercept_))
print("T4b: slope matches Task 3     :", np.isclose(model.coef_[0], b1))
print("T4b: intercept matches Task 3 :", np.isclose(model.intercept_, b0))

## Task 5 · Evaluate on the test set
- **(a)** Predict the test prices. Compute MAE, MSE, RMSE and R² **with the formulas** in NumPy.
- **(b)** Check MAE, MSE and R² with `sklearn.metrics`.
- **(c)** Train R² and test R². Is the model overfitting?
- **(d)** Baseline: predict the **training mean** for every test ad. Its MAE and R².

**Write (Markdown):** explain MAE and R² of your model in plain words, and how much better it is than the baseline.

In [ ]:
# T5: evaluation
y_pred = model.predict(X_test)
yt = y_test.to_numpy()
res = yt - y_pred

mae  = np.mean(np.abs(res))
mse  = np.mean(res ** 2)
rmse = np.sqrt(mse)
sse  = np.sum(res ** 2)
sst  = np.sum((yt - yt.mean()) ** 2)
r2   = 1 - sse / sst
print("T5a (formulas): MAE = %.4f lakh | MSE = %.4f lakh^2 | RMSE = %.4f lakh | R2 = %.4f" % (mae, mse, rmse, r2))

print("T5b (sklearn) : MAE = %.4f | MSE = %.4f | R2 = %.4f" % (
    mean_absolute_error(yt, y_pred), mean_squared_error(yt, y_pred), r2_score(yt, y_pred)))
print("T5b all match :", np.isclose(mae, mean_absolute_error(yt, y_pred)),
      np.isclose(mse, mean_squared_error(yt, y_pred)), np.isclose(r2, r2_score(yt, y_pred)))

r2_train = r2_score(y_train, model.predict(X_train))
print("T5c: train R2 = %.4f | test R2 = %.4f | gap = %.4f" % (r2_train, r2, r2_train - r2))
print("T5c: overfitting?", "No - gap is small" if abs(r2_train - r2) < 0.05 else "Possibly - gap is large")

base_pred = np.full_like(yt, y_train.mean(), dtype=float)
base_mae = mean_absolute_error(yt, base_pred)
base_r2 = r2_score(yt, base_pred)
print("T5d: baseline (train mean) MAE = %.4f lakh | R2 = %.4f" % (base_mae, base_r2))
print("T5d: model cuts MAE by %.1f%% versus the baseline" % (100 * (1 - mae / base_mae)))

**MAE** is the average size of the miss, in lakh: on a typical unseen ad the prediction is off by about this many lakh (multiply by 100,000 for rupees). **R²** is the share of the variation in prices that age removes compared with guessing the average price for every car. It is *not* "accuracy". The baseline always guesses the training mean, so its R² is about 0 and its MAE is large. The model's lower MAE (see the percentage printed in T5d) and its R² well above 0 show that age gives a real improvement. Train and test R² are close, so there is no sign of overfitting, which is expected for a two-parameter model on hundreds of ads.

## Task 6 · Visualise the line and the errors
- **(a)** One plot: training ads (grey), test ads, and the fitted line. Title, labels, legend.
- **(b)** A residual plot for the test set: residual (y − ŷ) against predicted price, with a line at 0.
- **(c)** The five test ads with the largest absolute error: `Year`, `KMs Driven`, `Registered City`, actual and predicted price.

**Write (Markdown):** does the residual plot show a curve or a funnel? Are the largest errors mistakes of the model or of the data?

In [ ]:
# T6: visualise
# (a) data + fitted line
grid = pd.DataFrame({"Age": np.linspace(gli["Age"].min(), gli["Age"].max(), 100)})
plt.figure(figsize=(7.5, 4.8))
plt.scatter(X_train["Age"], y_train, color="lightgrey", s=16, label="Train ads")
plt.scatter(X_test["Age"], y_test, color="tab:orange", s=18, label="Test ads")
plt.plot(grid["Age"], model.predict(grid), color="tab:blue", lw=2.5, label="Fitted line")
plt.title("Corolla GLI: fitted line on train and test ads")
plt.xlabel("Age (years)"); plt.ylabel("Price (lakh PKR)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig("fig_t6_line.png", dpi=150); plt.show()

# (b) residual plot
plt.figure(figsize=(7.5, 4.5))
plt.scatter(y_pred, res, alpha=0.6, s=18, color="tab:purple")
plt.axhline(0, color="black", lw=1.5)
plt.title("Residuals on the test set")
plt.xlabel("Predicted price (lakh PKR)"); plt.ylabel("Residual = actual - predicted (lakh)")
plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig("fig_t6_resid.png", dpi=150); plt.show()

# (c) five largest errors
show = [c for c in ["Year", "KMs Driven", "Registered City"] if c in gli.columns]
err = gli.loc[X_test.index, show].copy()
err["Actual"] = yt
err["Predicted"] = y_pred.round(2)
err["AbsError"] = np.abs(res).round(2)
top5 = err.sort_values("AbsError", ascending=False).head(5)
print("T6c: five largest test errors")
print(top5.to_string())

**Residual plot.** Look at the cloud around the zero line. If it is a flat, even band with no U-shape, the straight line fits (no curve). If it widens toward expensive cars, that is a funnel, meaning expensive (newer) cars are predicted less consistently. The largest errors in (c) are mostly **data problems rather than model mistakes**: unusual mileage, a different city, or a seller who mis-priced or inflated the ad. Age alone cannot know any of that, so those points reflect missing features and noisy ads, not a broken model. A few may also be listing errors.

## Task 7 · Use the model
- **(a)** Predict the price of a 2018, a 2015 and a 2010 Corolla GLI (age = 2020 − year), using a DataFrame with an `Age` column.
- **(b)** Predict the price of a 1995 Corolla GLI (age 25).

**Write (Markdown):** is the answer in (b) believable? Why not?

In [ ]:
# T7: use the model
new = pd.DataFrame({"Age": [2020 - 2018, 2020 - 2015, 2020 - 2010]})
p = model.predict(new)
for yr, a, v in zip([2018, 2015, 2010], new["Age"], p):
    print("T7a: %d Corolla GLI (age %d) -> %.2f lakh (about Rs %s)" % (yr, a, v, format(int(round(v * 1e5)), ",")))

p95 = model.predict(pd.DataFrame({"Age": [25]}))[0]
print("T7b: 1995 Corolla GLI (age 25) -> %.2f lakh  [training ages: %d to %d]" % (p95, gli["Age"].min(), gli["Age"].max()))

**Is (b) believable?** No. The model was trained on cars aged about 2 to 15, and 25 years is far outside that range, so this is **extrapolation**. The straight line keeps falling at the same rate forever, so it gives a price that is far too low (it can even go negative for ages far enough out). Real prices level off for old cars, and a 1995 car may even be a collectable. The data never showed what happens beyond the training range, so the model cannot know. Use the line only inside (or very near) the range of ages it saw.

## Task 8 · Reflection (Markdown cell)
Write 3–5 sentences: the steps you followed to build the model, why the model is judged on the test set and not the training set, which number you would tell a car buyer and why, and one limitation of using age alone.

I loaded and filtered the Corolla GLI cash ads, plotted price against age, split the data 80/20 with `random_state=42`, computed the least squares slope and intercept by hand, confirmed them with `LinearRegression`, and then evaluated MAE, MSE, RMSE and R² on the test ads against a mean-price baseline. The model is judged on the test set because those ads were never used for fitting, so the test score estimates how it will do on new ads, whereas the training score can be flattered by memorising the data. To a car buyer I would quote the **MAE in lakh** (e.g. "typically off by about X lakh"), not R², because it is in rupees and tells them how far off a price estimate may be, while R² is a unitless share of variation. The main limitation of age alone is that it ignores mileage, condition, city and seller, so two cars of the same age can differ by lakhs, and the line cannot be trusted outside the ages seen in training.